# Evaluación robusta de clasificadores
## Notebook 07 — Validación cruzada, métricas por clase y curvas ROC

Este notebook complementa el notebook 06. Toma los mismos datos y clasificadores
para las **cuatro técnicas** (STFT, CWT, SST, WPD), pero reemplaza el split único
por evaluaciones más rigurosas y corrige el sesgo de distribución entre train y test.

---

## Por qué el split de bloques alternos del notebook 06 tiene un problema

El split asignaba bloques pares a train e impares a test, generando:

| Conjunto | Clase 0 (ojos abiertos) | Clase 1 (ojos cerrados) |
|----------|------------------------|------------------------|
| Train    | 45.8 %                 | 54.2 %                 |
| Test     | **62.9 %**             | **37.1 %**             |

Un clasificador que prediga siempre 'ojos abiertos' obtiene 45.8 % en train
pero 62.9 % en test — sin aprender nada. Eso puede hacer que un modelo malo
parezca que 'mejora' al pasar de train a test.

## Soluciones implementadas

1. **Balanced Accuracy** — promedia el recall de cada clase, eliminando el efecto
   del desbalance. Un clasificador trivial siempre obtiene BAcc = 0.5.

2. **TimeSeriesSplit (5-fold)** — repite la evaluación con ventanas temporales
   crecientes. El resultado es `media ± std`, permitiendo saber si las diferencias
   entre técnicas son consistentes o producto de una partición afortunada.

3. **Curvas ROC** — muestra el tradeoff completo entre sensibilidad y especificidad
   para las 4 técnicas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
warnings.filterwarnings('ignore')

# Paleta fija para las 4 técnicas
COLORS = {
    'STFT': '#2980b9',
    'CWT':  '#c0392b',
    'SST':  '#27ae60',
    'WPD':  '#e67e22',
}
TECNICAS = ['STFT', 'CWT', 'SST', 'WPD']

# Cargar los 4 datasets
features_stft = pd.read_csv('data/features/features_stft.csv')
features_cwt  = pd.read_csv('data/features/features_cwt.csv')
features_sst  = pd.read_csv('data/features/features_sst.csv')
features_wpd  = pd.read_csv('data/features/features_wpd.csv')

datasets = {
    'STFT': features_stft,
    'CWT':  features_cwt,
    'SST':  features_sst,
    'WPD':  features_wpd,
}

Xs = {name: df.drop('label', axis=1).values for name, df in datasets.items()}
ys = {name: df['label'].values             for name, df in datasets.items()}

feature_names      = features_stft.columns[:-1].tolist()
features_por_canal = ['alpha_abs', 'alpha_rel', 'entropy', 'cog']
canales_todos = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']
canales_alfa  = ['O1','O2','P7','P8']

def get_feature_indices(feat_names, canales):
    idxs = []
    for canal in canales:
        for feat in features_por_canal:
            col = f'{canal}_{feat}'
            if col in feat_names:
                idxs.append(feat_names.index(col))
    return idxs

idx_config_a = get_feature_indices(feature_names, canales_todos)
idx_config_b = get_feature_indices(feature_names, canales_alfa)

print('Datasets cargados:')
for name, X in Xs.items():
    print(f'  {name}: {X.shape}, clase 0={np.sum(ys[name]==0)}, clase 1={np.sum(ys[name]==1)}')
print(f'Config A: {len(idx_config_a)} features | Config B: {len(idx_config_b)} features')


In [ ]:
## Split único temporal (igual que en notebook 06)

# Se conserva el mismo split para que la tabla resumen pueda comparar
# split único vs cross-validación en las mismas condiciones.
n_samples  = Xs['STFT'].shape[0]
block_size = 30
block_numbers = np.arange(n_samples) // block_size
train_idx = np.where(block_numbers % 2 == 0)[0]
test_idx  = np.where(block_numbers % 2 == 1)[0]

# Construir splits para las 4 técnicas
splits = {}
for name in TECNICAS:
    X, y = Xs[name], ys[name]
    sc_a, sc_b = StandardScaler(), StandardScaler()
    Xtr_a = X[train_idx][:, idx_config_a]
    Xte_a = X[test_idx ][:, idx_config_a]
    Xtr_b = X[train_idx][:, idx_config_b]
    Xte_b = X[test_idx ][:, idx_config_b]
    splits[name] = {
        'X_tr_a': sc_a.fit_transform(Xtr_a), 'X_te_a': sc_a.transform(Xte_a),
        'X_tr_b': sc_b.fit_transform(Xtr_b), 'X_te_b': sc_b.transform(Xte_b),
        'y_tr':   y[train_idx], 'y_te': y[test_idx],
    }

print(f'Train: {len(train_idx)} muestras  |  Test: {len(test_idx)} muestras')
s0 = splits['STFT']
print(f'Distribucion train: clase 0 = {np.mean(s0["y_tr"]==0):.1%}')
print(f'Distribucion test:  clase 0 = {np.mean(s0["y_te"]==0):.1%}')
print('\nAtencion: diferencia de distribucion entre train y test -> ver Seccion 1 (Balanced Accuracy).')


In [ ]:
## Definir clasificadores y entrenar split único (24 combinaciones)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
}

def train_and_evaluate(X_train, X_test, y_train, y_test, model):
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    return {
        'model':   model,
        'y_pred':  y_pred,
        'y_proba': y_proba,
        'acc':     accuracy_score(y_test, y_pred),
        'bacc':    balanced_accuracy_score(y_test, y_pred),
        'f1':      f1_score(y_test, y_pred, average='macro'),
        'auc':     roc_auc_score(y_test, y_proba) if y_proba is not None else float('nan'),
        'cm':      confusion_matrix(y_test, y_pred),
    }

results = []
print('Entrenando 24 combinaciones (4 tecnicas x 2 configs x 3 clasificadores)...\n')

for tecnica in TECNICAS:
    s = splits[tecnica]
    for cfg_name, X_tr, X_te in [
        ('A (56 features)', s['X_tr_a'], s['X_te_a']),
        ('B (16 features)', s['X_tr_b'], s['X_te_b']),
    ]:
        for clf_name, clf in classifiers.items():
            r = train_and_evaluate(X_tr, X_te, s['y_tr'], s['y_te'], clone(clf))
            results.append({'Técnica': tecnica, 'Config': cfg_name, 'Clasificador': clf_name, **r})
            print(f'  {tecnica} {cfg_name} | {clf_name:20s} '
                  f'Acc: {r["acc"]:.3f}  BAcc: {r["bacc"]:.3f}  '
                  f'F1: {r["f1"]:.3f}  AUC: {r["auc"]:.3f}')

print('\nEntrenamiento completado.')


## Sección 1: Balanced Accuracy — corrigiendo el sesgo de distribución

La **Accuracy estándar** es sensible a la distribución de clases. Con 62.9 % de clase 0 en el test:
- Un clasificador que prediga siempre "ojos abiertos" (clase 0) obtiene **62.9 % de accuracy** sin aprender.
- El mismo clasificador obtiene solo **45.8 %** en el conjunto de entrenamiento.
- Eso da la apariencia de que el modelo "mejora de train a test", cuando en realidad es peor.

La **Balanced Accuracy** promedia el recall de cada clase:

$$\text{BAcc} = \frac{\text{Sensibilidad} + \text{Especificidad}}{2} = \frac{\text{TPR} + \text{TNR}}{2}$$

Al promediar los recall, normaliza la contribución de cada clase por su tamaño real.
Un clasificador que prediga siempre clase 0 obtiene **BAcc = 0.5** (aleatorio) en cualquier distribución,
mientras que la accuracy estándar varía según el desbalance del conjunto evaluado.

In [ ]:
## Balanced Accuracy: comparación con Accuracy estándar (24 combinaciones)

rows = []
for r in results:
    tn, fp = r['cm'][0,0], r['cm'][0,1]
    fn, tp = r['cm'][1,0], r['cm'][1,1]
    rows.append({
        'Técnica':      r['Técnica'],
        'Config':       r['Config'],
        'Clasificador': r['Clasificador'],
        'Acc':          round(r['acc'],  4),
        'BAcc':         round(r['bacc'], 4),
        'Delta':        round(r['bacc'] - r['acc'], 4),
        'Sens (C1)':    round(tp/(tp+fn) if (tp+fn)>0 else 0, 4),
        'Spec (C0)':    round(tn/(tn+fp) if (tn+fp)>0 else 0, 4),
    })

bacc_df = pd.DataFrame(rows)
print(bacc_df.to_string(index=False))

print('\n--- Ranking por BAcc promedio por tecnica ---')
ranking = bacc_df.groupby('Técnica')['BAcc'].mean().loc[TECNICAS].sort_values(ascending=False)
for i, (tecnica, val) in enumerate(ranking.items(), 1):
    print(f'  {i}. {tecnica}: BAcc promedio = {val:.4f}')

print('\n--- Modelos con Sensibilidad < 0.4 (posible sesgo hacia clase 0) ---')
sesgados = bacc_df[bacc_df['Sens (C1)'] < 0.4]
if len(sesgados) > 0:
    print(sesgados[['Técnica','Config','Clasificador','Acc','BAcc','Sens (C1)','Spec (C0)']].to_string(index=False))
else:
    print('Ninguno -- todos los modelos tienen recall minimo en ambas clases.')


## Sección 2: Cross-validación temporal (TimeSeriesSplit, 5 folds)

El split único entrega **una sola estimación** con alta varianza: si ese test particular fuera
más fácil o difícil que el promedio, las conclusiones serían incorrectas. Con solo 236 muestras,
esto es especialmente probable.

`TimeSeriesSplit` con K=5 divide el dataset en ventanas temporales crecientes:

| Fold | Train       | Test        |
|------|-------------|-------------|
| 1    | muestras 1–40   | 41–79  |
| 2    | muestras 1–79   | 80–118 |
| 3    | muestras 1–118  | 119–157|
| 4    | muestras 1–157  | 158–196|
| 5    | muestras 1–196  | 197–236|

Siempre se entrena con pasado y se testea con futuro — se respeta la estructura temporal
sin mezclar información de distintos momentos. El escalado va **dentro de cada fold**
para evitar que las estadísticas del test contaminen el ajuste del scaler.

In [ ]:
## Cross-validación temporal con Balanced Accuracy (4 técnicas)

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

cv_configs = []
for tecnica in TECNICAS:
    X = Xs[tecnica]
    y = ys[tecnica]
    cv_configs += [
        (tecnica, X[:, idx_config_a], y, 'A (56 features)'),
        (tecnica, X[:, idx_config_b], y, 'B (16 features)'),
    ]

cv_results = []
print(f'Cross-validacion temporal (TimeSeriesSplit, {N_SPLITS} folds)\n')

for tecnica, X_full, y_full, config_name in cv_configs:
    for clf_name, clf_template in classifiers.items():
        fold_acc, fold_bacc, fold_f1, fold_auc = [], [], [], []

        for tr_idx, te_idx in tscv.split(X_full):
            X_tr, X_te = X_full[tr_idx], X_full[te_idx]
            y_tr, y_te = y_full[tr_idx], y_full[te_idx]

            sc = StandardScaler()
            X_tr_sc = sc.fit_transform(X_tr)
            X_te_sc = sc.transform(X_te)

            clf_fold = clone(clf_template)
            clf_fold.fit(X_tr_sc, y_tr)
            y_pred  = clf_fold.predict(X_te_sc)
            y_proba = clf_fold.predict_proba(X_te_sc)[:, 1] if hasattr(clf_fold, 'predict_proba') else None

            fold_acc.append(accuracy_score(y_te, y_pred))
            fold_bacc.append(balanced_accuracy_score(y_te, y_pred))
            fold_f1.append(f1_score(y_te, y_pred, average='macro'))
            if y_proba is not None and len(np.unique(y_te)) > 1:
                fold_auc.append(roc_auc_score(y_te, y_proba))

        cv_results.append({
            'Técnica':    tecnica,
            'Config':     config_name,
            'Clasificador': clf_name,
            'Acc_mean':  np.mean(fold_acc),  'Acc_std':  np.std(fold_acc),
            'BAcc_mean': np.mean(fold_bacc), 'BAcc_std': np.std(fold_bacc),
            'F1_mean':   np.mean(fold_f1),   'F1_std':   np.std(fold_f1),
            'AUC_mean':  np.mean(fold_auc) if fold_auc else float('nan'),
            'AUC_std':   np.std(fold_auc)  if fold_auc else float('nan'),
        })
        auc_str = f'AUC {np.mean(fold_auc):.3f}+-{np.std(fold_auc):.3f}' if fold_auc else 'AUC N/A'
        print(f'{tecnica} {config_name} | {clf_name:20s}: '
              f'Acc {np.mean(fold_acc):.3f}+-{np.std(fold_acc):.3f}  '
              f'BAcc {np.mean(fold_bacc):.3f}+-{np.std(fold_bacc):.3f}  '
              f'F1 {np.mean(fold_f1):.3f}+-{np.std(fold_f1):.3f}  {auc_str}')

cv_df = pd.DataFrame(cv_results)

print('\n--- Ranking por BAcc_mean promedio por tecnica (CV) ---')
cv_ranking = cv_df.groupby('Técnica')['BAcc_mean'].mean().loc[TECNICAS].sort_values(ascending=False)
for i, (tecnica, val) in enumerate(cv_ranking.items(), 1):
    print(f'  {i}. {tecnica}: BAcc_mean promedio = {val:.4f}')

print('\nCross-validacion completada.')


## Sección 3: Curvas ROC comparativas

Un único valor AUC no muestra si las curvas de las 4 técnicas se cruzan.
Si se cruzan, ninguna técnica domina en todos los umbrales.

Convención: **color por técnica** (azul=STFT, rojo=CWT, verde=SST, naranja=WPD),
**trazo sólido** = Config A (56 features), **trazo punteado** = Config B (16 features).


In [ ]:
## Curvas ROC: las 4 técnicas por clasificador

linestyles = {'A (56 features)': '-', 'B (16 features)': '--'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Curvas ROC por Clasificador -- solido=Config A, punteado=Config B',
             fontsize=12, fontweight='bold')

for clf_idx, clf_name in enumerate(list(classifiers.keys())):
    ax = axes[clf_idx]

    for tecnica in TECNICAS:
        for config in ['A (56 features)', 'B (16 features)']:
            r = next((x for x in results if x['Técnica'] == tecnica
                      and x['Config'] == config
                      and x['Clasificador'] == clf_name), None)
            if r is None or r['y_proba'] is None:
                continue
            fpr, tpr, _ = roc_curve(splits[tecnica]['y_te'], r['y_proba'])
            label = f"{tecnica} {config.split('(')[0].strip()} (AUC={r['auc']:.3f})"
            ax.plot(fpr, tpr, color=COLORS[tecnica],
                    linestyle=linestyles[config], linewidth=2, label=label, alpha=0.85)

    ax.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.4, label='Azar')
    ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=9)
    ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=9)
    ax.set_title(clf_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()


## Sección 4: Tabla resumen — split único vs cross-validación

Esta tabla cruza ambas evaluaciones para responder:

- **¿El split único fue representativo?** Si `Acc(split) ≈ Acc_mean(CV)`, sí.
  Si `Acc(split)` es significativamente mayor que `Acc_mean(CV)`, el split único
  era optimista (ese fold particular era más fácil que el promedio).

- **¿Los resultados son estables?** Un `std` alto en CV indica que la performance
  depende mucho de qué parte del registro se usa como test — importante
  declararlo en el informe.

- **La columna BAcc(CV)** es la métrica más confiable del notebook porque combina
  corrección de distribución (balanced) con evaluación múltiple (CV).

In [ ]:
## Tabla resumen final: split único vs cross-validación (24 combinaciones)

results_df_num = pd.DataFrame([
    {'Técnica': r['Técnica'], 'Config': r['Config'], 'Clasificador': r['Clasificador'],
     'Acc': r['acc'], 'BAcc': r['bacc'], 'F1': r['f1'], 'AUC': r['auc']}
    for r in results
])

summary_rows = []
for _, row_cv in cv_df.iterrows():
    row_sp = results_df_num[
        (results_df_num['Técnica']      == row_cv['Técnica']) &
        (results_df_num['Config']       == row_cv['Config']) &
        (results_df_num['Clasificador'] == row_cv['Clasificador'])
    ].iloc[0]
    summary_rows.append({
        'Técnica':      row_cv['Técnica'],
        'Config':       row_cv['Config'],
        'Clasificador': row_cv['Clasificador'],
        'Acc (split)':  f"{row_sp['Acc']:.3f}",
        'BAcc (split)': f"{row_sp['BAcc']:.3f}",
        'Acc (CV)':     f"{row_cv['Acc_mean']:.3f}+-{row_cv['Acc_std']:.3f}",
        'BAcc (CV)':    f"{row_cv['BAcc_mean']:.3f}+-{row_cv['BAcc_std']:.3f}",
        'F1 (CV)':      f"{row_cv['F1_mean']:.3f}+-{row_cv['F1_std']:.3f}",
        'AUC (CV)':     f"{row_cv['AUC_mean']:.3f}+-{row_cv['AUC_std']:.3f}",
    })

summary_df = pd.DataFrame(summary_rows)
print('=' * 115)
print('TABLA RESUMEN: Split unico vs Cross-validacion (5-fold TimeSeriesSplit) -- 24 combinaciones')
print('=' * 115)
print(summary_df.to_string(index=False))

print('\n--- Ranking final por BAcc (CV) promedio por tecnica ---')
ranking_final = cv_df.groupby('Técnica')['BAcc_mean'].mean().loc[TECNICAS].sort_values(ascending=False)
for i, (tecnica, val) in enumerate(ranking_final.items(), 1):
    print(f'  {i}. {tecnica}: BAcc_mean = {val:.4f}')

print('\n--- Mejor combinacion individual segun BAcc (CV) ---')
best_cv = cv_df.loc[cv_df['BAcc_mean'].idxmax()]
print(f"{best_cv['Técnica']} + {best_cv['Config']} + {best_cv['Clasificador']}: "
      f"BAcc {best_cv['BAcc_mean']:.3f}+-{best_cv['BAcc_std']:.3f}, "
      f"AUC {best_cv['AUC_mean']:.3f}+-{best_cv['AUC_std']:.3f}")
